In [1]:
# environment: /glade/work/dkimpara/conda-envs/xesmf

In [1]:
import xarray as xr
import shutil

import os
from os.path import join
import glob
import numpy as np
import datetime

from joblib import Parallel, delayed
import joblib

from glob import glob

from functools import partial
import dask.array as da

import pandas as pd
import tqdm


In [5]:
top_dir = "/glade/derecho/scratch/dkimpara/goes-cloud-dataset"

channels = [4, 7, 8, 9, 10, 13]
zarr_path = "/glade/derecho/scratch/dkimpara/goes-cloud-dataset/goes_10km.zarr"

test = 1

num_cpus = 2

In [6]:
zarr_ds = xr.open_dataset(zarr_path, consolidated=False).drop_duplicates(
        dim="t"
    ).sortby(
        "t"
    )
if test:
    print("testing")
    zarr_ds = zarr_ds.isel(t=slice(2, 6))

# check zarr

In [5]:
def compute_std_dev(indices):
    mean_ds = xr.open_dataset(join("/glade/derecho/scratch/dkimpara/goes-cloud-dataset/", "minmaxmean.nc"), engine="h5netcdf")
    mean_da = mean_ds["mean"]

    zarr_ds = xr.open_dataset("/glade/derecho/scratch/dkimpara/goes-cloud-dataset/goes_10km.zarr", consolidated=False)

    first_index = indices[0]

    ds = zarr_ds.isel(t=first_index)
    sum_square_diff = ((ds.BT_or_R - mean_da) ** 2).mean(dim=["latitude", "longitude"], skipna=True)

    for i in indices[1:]:
        ds = zarr_ds.isel(t=i)
        sum_square_diff += ((ds.BT_or_R - mean_da) ** 2).mean(dim=["latitude", "longitude"], skipna=True)

    return sum_square_diff / len(indices)



def minmaxmean(indices):
    zarr_ds = xr.open_dataset("/glade/derecho/scratch/dkimpara/goes-cloud-dataset/goes_10km.zarr", consolidated=False).drop_duplicates(
        dim="t"
    ).sortby(
        "t"
    )

    first_index = indices[0]
    
    da = zarr_ds.isel(t=first_index).BT_or_R

    sum_means = da.mean(dim=["latitude", "longitude"], skipna=True)
    maximum = [da.max(dim=["latitude", "longitude"])]
    minimum = [da.min(dim=["latitude", "longitude"])]
    
    for i in indices[1:]:
        da = zarr_ds.isel(t=i).BT_or_R

        sum_means += da.mean(dim=["latitude", "longitude"], skipna=True)

        maximum.append(da.max(dim=["latitude", "longitude"]))
        minimum.append(da.min(dim=["latitude", "longitude"]))

    means = sum_means / len(indices)
    maximum = xr.concat(maximum, dim="t").max(dim="t")
    minimum = xr.concat(minimum, dim="t").min(dim="t")

    return minimum, maximum, means

def chunk_list(a, num_chunks):
    step = len(a) // num_chunks
    return [a[i:i + step] for i in range(0, len(a), step)]



In [6]:
indices = list(range(len(zarr_ds.t)))
# indices = list(range(2,6))

chunked = chunk_list(indices, num_cpus - 1)

if test:
    chunked = [indices[ :2], indices[2:]]

In [7]:
results = Parallel(n_jobs = num_cpus - 1)(delayed(minmaxmean)(index_list)
                            for index_list in chunked)

minimums, maximums, means = zip(*results)
out_ds = xr.Dataset({
    "min": ("channel", np.min(minimums, axis=0)),
    "max": ("channel", np.max(maximums, axis=0)),
    "mean": ("channel", np.nanmean(means, axis=0)),
})
out_ds.to_netcdf(join(top_dir, "minmaxmean.nc"), engine="h5netcdf")

In [9]:
results = Parallel(n_jobs = num_cpus - 1)(delayed(compute_std_dev)(index_list)
                            for index_list in chunked)

ds = xr.open_dataset(join(top_dir, "minmaxmean.nc"), engine="h5netcdf")

ds["std"] = ("channel", np.sqrt(np.nanmean(results,axis=0)))

ds.to_netcdf(join(top_dir, "data_stats.nc"), engine="h5netcdf")

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '/glade/derecho/scratch/dkimpara/goes-cloud-dataset/minmaxmean.nc', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)